In [1]:
#!/usr/bin/env python3
"""
fall_detection_1dcnn.py

Testprojekt: 1D-CNN für Sturzdetektion mit synthetischen Android-Sensordaten.

Das Skript macht vier Dinge:
1. synthetische Sensordaten erzeugen
2. ein kleines 1D-CNN trainieren
3. Modell evaluieren
4. Modell als TensorFlow Lite Datei für Android exportieren

Eingabeformat pro Beispiel:
    100 Zeitschritte × 6 Kanäle

Kanäle:
    ax, ay, az, gx, gy, gz

Annahme:
    50 Hz Samplingrate
    2 Sekunden Fensterlänge

Installation:
    pip install numpy tensorflow matplotlib

Ausführen:
    python fall_detection_1dcnn.py
"""

'\nfall_detection_1dcnn.py\n\nTestprojekt: 1D-CNN für Sturzdetektion mit synthetischen Android-Sensordaten.\n\nDas Skript macht vier Dinge:\n1. synthetische Sensordaten erzeugen\n2. ein kleines 1D-CNN trainieren\n3. Modell evaluieren\n4. Modell als TensorFlow Lite Datei für Android exportieren\n\nEingabeformat pro Beispiel:\n    100 Zeitschritte × 6 Kanäle\n\nKanäle:\n    ax, ay, az, gx, gy, gz\n\nAnnahme:\n    50 Hz Samplingrate\n    2 Sekunden Fensterlänge\n\nInstallation:\n    pip install numpy tensorflow matplotlib\n\nAusführen:\n    python fall_detection_1dcnn.py\n'

In [2]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf


In [3]:
# ------------------------------------------------------------
# Konfiguration
# ------------------------------------------------------------

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SAMPLE_RATE_HZ = 50
WINDOW_SECONDS = 2.0
WINDOW_SIZE = int(SAMPLE_RATE_HZ * WINDOW_SECONDS)
N_CHANNELS = 6

# ax, ay, az in m/s²; gx, gy, gz in rad/s
CHANNEL_NAMES = ["ax", "ay", "az", "gx", "gy", "gz"]

CLASS_NAMES = [
    "alltag_ruhend",
    "gehen",
    "hinlegen",
    "handy_faellt",
    "sturz",
]

N_CLASSES = len(CLASS_NAMES)
EXAMPLES_PER_CLASS = 1200

OUTPUT_DIR = Path("fall_detection_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
# -----------------------------------------------------------
# Hilfsfunktionen für synthetische Sensordaten
# ------------------------------------------------------------

def random_unit_vector() -> np.ndarray:
    """Erzeugt eine zufällige 3D-Einheitsrichtung."""
    v = np.random.normal(size=3)
    norm = np.linalg.norm(v)
    if norm < 1e-8:
        return np.array([0.0, 0.0, 1.0], dtype=np.float32)
    return (v / norm).astype(np.float32)


def smooth_transition(start: np.ndarray, end: np.ndarray, n: int) -> np.ndarray:
    """Weicher Übergang zwischen zwei 3D-Vektoren."""
    if n <= 1:
        return np.repeat(end[None, :], max(n, 1), axis=0)

    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    # Smoothstep: 3t² - 2t³
    s = 3 * t**2 - 2 * t**3
    return (1.0 - s[:, None]) * start[None, :] + s[:, None] * end[None, :]


def add_sensor_noise(data: np.ndarray, acc_noise: float = 0.25, gyro_noise: float = 0.03) -> np.ndarray:
    """Fügt Messrauschen hinzu."""
    noisy = data.copy()
    noisy[:, 0:3] += np.random.normal(0.0, acc_noise, size=(WINDOW_SIZE, 3))
    noisy[:, 3:6] += np.random.normal(0.0, gyro_noise, size=(WINDOW_SIZE, 3))
    return noisy.astype(np.float32)


def empty_window() -> np.ndarray:
    return np.zeros((WINDOW_SIZE, N_CHANNELS), dtype=np.float32)


def generate_idle() -> np.ndarray:
    """Ruhige Alltagssituation: Handy liegt oder steckt ruhig in Tasche."""
    data = empty_window()
    gravity_dir = random_unit_vector()
    gravity = 9.81 * gravity_dir

    data[:, 0:3] = gravity[None, :]
    data[:, 3:6] = np.random.normal(0.0, 0.02, size=(WINDOW_SIZE, 3))

    return add_sensor_noise(data, acc_noise=0.12, gyro_noise=0.015)


def generate_walking() -> np.ndarray:
    """Gehen: periodische Beschleunigung und leichte Rotation."""
    data = empty_window()
    t = np.arange(WINDOW_SIZE) / SAMPLE_RATE_HZ

    gravity_dir = np.array([0.0, 0.0, 1.0], dtype=np.float32)
    gravity = 9.81 * gravity_dir

    step_freq = np.random.uniform(1.4, 2.4)
    phase = np.random.uniform(0.0, 2.0 * math.pi)
    amp_vertical = np.random.uniform(0.8, 2.4)
    amp_side = np.random.uniform(0.2, 0.8)

    vertical = amp_vertical * np.sin(2.0 * math.pi * step_freq * t + phase)
    side_x = amp_side * np.sin(2.0 * math.pi * step_freq * t + phase + 1.0)
    side_y = amp_side * np.cos(2.0 * math.pi * step_freq * t + phase)

    data[:, 0] = gravity[0] + side_x
    data[:, 1] = gravity[1] + side_y
    data[:, 2] = gravity[2] + vertical

    gyro_amp = np.random.uniform(0.15, 0.7)
    data[:, 3] = gyro_amp * np.sin(2.0 * math.pi * step_freq * t + phase)
    data[:, 4] = gyro_amp * np.cos(2.0 * math.pi * step_freq * t + phase + 0.6)
    data[:, 5] = 0.3 * gyro_amp * np.sin(2.0 * math.pi * step_freq * t + phase + 1.2)

    return add_sensor_noise(data, acc_noise=0.25, gyro_noise=0.05)


def generate_lying_down() -> np.ndarray:
    """Normales Hinlegen: deutliche Lageänderung, aber ohne harten Aufprallpeak."""
    data = empty_window()

    start_dir = np.array([0.0, 0.0, 1.0], dtype=np.float32)
    end_dir = random_unit_vector()

    start = np.random.randint(15, 45)
    duration = np.random.randint(30, 55)
    end = min(start + duration, WINDOW_SIZE)

    before = np.repeat(start_dir[None, :], start, axis=0)
    during = smooth_transition(start_dir, end_dir, end - start)
    after = np.repeat(end_dir[None, :], WINDOW_SIZE - end, axis=0)
    orientation = np.vstack([before, during, after])[:WINDOW_SIZE]

    data[:, 0:3] = 9.81 * orientation

    # Während des Hinlegens mäßige Rotation.
    data[start:end, 3:6] = np.random.normal(0.0, 0.8, size=(end - start, 3))

    # Kein starker Impact, nur kleine Beschleunigungswelle.
    if end < WINDOW_SIZE:
        bump_center = min(end, WINDOW_SIZE - 1)
        bump_width = np.random.randint(5, 12)
        bump_amp = np.random.uniform(1.0, 2.8)
        for i in range(max(0, bump_center - bump_width), min(WINDOW_SIZE, bump_center + bump_width)):
            weight = math.exp(-((i - bump_center) ** 2) / (2.0 * (bump_width / 2.0) ** 2))
            data[i, 2] += bump_amp * weight

    return add_sensor_noise(data, acc_noise=0.22, gyro_noise=0.06)


def generate_phone_drop() -> np.ndarray:
    """Handy fällt alleine: freier Fall, Aufprall, danach sehr ruhig."""
    data = empty_window()

    start_dir = random_unit_vector()
    rest_dir = random_unit_vector()

    drop_start = np.random.randint(20, 45)
    free_fall_len = np.random.randint(8, 18)
    impact_idx = min(drop_start + free_fall_len, WINDOW_SIZE - 5)

    # Vor dem Drop normaler Gravitationseinfluss.
    data[:drop_start, 0:3] = 9.81 * start_dir

    # Freier Fall: Beschleunigung nahe 0, starke Rotation möglich.
    data[drop_start:impact_idx, 0:3] = np.random.normal(0.0, 0.5, size=(impact_idx - drop_start, 3))
    data[drop_start:impact_idx, 3:6] = np.random.normal(0.0, 5.0, size=(impact_idx - drop_start, 3))

    # Aufprall: kurzer sehr hoher Peak.
    impact_amp = np.random.uniform(22.0, 45.0)
    impact_dir = random_unit_vector()
    data[impact_idx, 0:3] = impact_amp * impact_dir
    data[impact_idx, 3:6] = np.random.normal(0.0, 4.0, size=3)

    # Nach dem Aufprall liegt das Handy ruhig.
    data[impact_idx + 1 :, 0:3] = 9.81 * rest_dir
    data[impact_idx + 1 :, 3:6] = np.random.normal(0.0, 0.03, size=(WINDOW_SIZE - impact_idx - 1, 3))

    return add_sensor_noise(data, acc_noise=0.18, gyro_noise=0.04)


def generate_fall() -> np.ndarray:
    """Personensturz: Bewegung, Fallphase, Aufprall, danach wenig Bewegung in neuer Lage."""
    data = empty_window()

    upright_dir = np.array([0.0, 0.0, 1.0], dtype=np.float32)
    lying_dir = random_unit_vector()

    fall_start = np.random.randint(25, 55)
    fall_duration = np.random.randint(15, 28)
    impact_idx = min(fall_start + fall_duration, WINDOW_SIZE - 8)

    # Vorher leichte Bewegung, etwa Gehen/Stehen.
    t = np.arange(WINDOW_SIZE) / SAMPLE_RATE_HZ
    pre = np.arange(fall_start)
    data[:fall_start, 0:3] = 9.81 * upright_dir
    data[pre, 2] += np.random.uniform(0.4, 1.4) * np.sin(2.0 * math.pi * np.random.uniform(1.0, 2.0) * t[pre])
    data[:fall_start, 3:6] = np.random.normal(0.0, 0.25, size=(fall_start, 3))

    # Fallphase: Orientierung ändert sich, Beschleunigung teils reduziert, Gyro hoch.
    orientation = smooth_transition(upright_dir, lying_dir, impact_idx - fall_start)
    data[fall_start:impact_idx, 0:3] = 9.81 * orientation * np.random.uniform(0.15, 0.65)
    data[fall_start:impact_idx, 3:6] = np.random.normal(0.0, 2.0, size=(impact_idx - fall_start, 3))

    # Impact: Peak, aber meist nicht so extrem wie reiner Phone-Drop.
    impact_amp = np.random.uniform(18.0, 36.0)
    impact_dir = random_unit_vector()
    data[impact_idx, 0:3] = impact_amp * impact_dir
    data[impact_idx, 3:6] = np.random.normal(0.0, 2.5, size=3)

    # Nachphase: Person liegt relativ ruhig, geringe Restbewegung.
    data[impact_idx + 1 :, 0:3] = 9.81 * lying_dir
    data[impact_idx + 1 :, 3:6] = np.random.normal(0.0, 0.08, size=(WINDOW_SIZE - impact_idx - 1, 3))

    return add_sensor_noise(data, acc_noise=0.22, gyro_noise=0.06)


def generate_example(class_id: int) -> np.ndarray:
    if class_id == 0:
        return generate_idle()
    if class_id == 1:
        return generate_walking()
    if class_id == 2:
        return generate_lying_down()
    if class_id == 3:
        return generate_phone_drop()
    if class_id == 4:
        return generate_fall()
    raise ValueError(f"Unbekannte Klasse: {class_id}")


def generate_dataset(examples_per_class: int) -> Tuple[np.ndarray, np.ndarray]:
    xs: List[np.ndarray] = []
    ys: List[int] = []

    for class_id, class_name in enumerate(CLASS_NAMES):
        print(f"Erzeuge Klasse {class_id}: {class_name}")
        for _ in range(examples_per_class):
            xs.append(generate_example(class_id))
            ys.append(class_id)

    x = np.stack(xs).astype(np.float32)
    y = np.array(ys, dtype=np.int64)

    indices = np.random.permutation(len(y))
    return x[indices], y[indices]


In [5]:
# ------------------------------------------------------------
# Datenaufteilung und Normalisierung
# ------------------------------------------------------------

def split_dataset(
    x: np.ndarray,
    y: np.ndarray,
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = len(y)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    x_train = x[:n_train]
    y_train = y[:n_train]

    x_val = x[n_train : n_train + n_val]
    y_val = y[n_train : n_train + n_val]

    x_test = x[n_train + n_val :]
    y_test = y[n_train + n_val :]

    return x_train, y_train, x_val, y_val, x_test, y_test


def compute_normalization(x_train: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Mittelwert und Standardabweichung pro Kanal berechnen."""
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True)
    std = np.maximum(std, 1e-6)
    return mean.astype(np.float32), std.astype(np.float32)


def normalize(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)

In [6]:
# ------------------------------------------------------------
# Modell
# ------------------------------------------------------------

def build_model() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(WINDOW_SIZE, N_CHANNELS), name="sensor_window")

    x = tf.keras.layers.Conv1D(32, kernel_size=5, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
    x = tf.keras.layers.Dropout(0.15)(x)

    x = tf.keras.layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
    x = tf.keras.layers.Dropout(0.20)(x)

    x = tf.keras.layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    outputs = tf.keras.layers.Dense(N_CLASSES, activation="softmax", name="class_probabilities")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [7]:
# ------------------------------------------------------------
# Evaluation und Export
# ------------------------------------------------------------

def plot_history(history: tf.keras.callbacks.History) -> None:
    plt.figure()
    plt.plot(history.history["accuracy"], label="train_accuracy")
    plt.plot(history.history["val_accuracy"], label="val_accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_accuracy.png", dpi=150)
    plt.close()

    plt.figure()
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=150)
    plt.close()


def confusion_matrix_np(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for true_id, pred_id in zip(y_true, y_pred):
        cm[int(true_id), int(pred_id)] += 1
    return cm


def plot_confusion_matrix(cm: np.ndarray) -> None:
    plt.figure(figsize=(8, 7))
    plt.imshow(cm)
    plt.title("Confusion Matrix")
    plt.xlabel("Vorhergesagte Klasse")
    plt.ylabel("Echte Klasse")
    plt.xticks(range(N_CLASSES), CLASS_NAMES, rotation=45, ha="right")
    plt.yticks(range(N_CLASSES), CLASS_NAMES)

    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
    plt.close()


def save_metadata(mean: np.ndarray, std: np.ndarray) -> None:
    metadata: Dict[str, object] = {
        "sample_rate_hz": SAMPLE_RATE_HZ,
        "window_seconds": WINDOW_SECONDS,
        "window_size": WINDOW_SIZE,
        "channels": CHANNEL_NAMES,
        "classes": CLASS_NAMES,
        "normalization": {
            "mean": mean.reshape(-1).tolist(),
            "std": std.reshape(-1).tolist(),
        },
        "input_shape": [1, WINDOW_SIZE, N_CHANNELS],
        "output_shape": [1, N_CLASSES],
        "note": "Android muss dieselbe Normalisierung verwenden: normalized = (raw - mean) / std",
    }

    with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)


def export_tflite(model: tf.keras.Model) -> None:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Float32-TFLite: am einfachsten für den Start.
    tflite_model = converter.convert()
    with open(OUTPUT_DIR / "fall_detection_1dcnn_float32.tflite", "wb") as f:
        f.write(tflite_model)

    # Optional: dynamisch quantisierte Version, kleiner und oft schneller.
    converter_optimized = tf.lite.TFLiteConverter.from_keras_model(model)
    converter_optimized.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quant_model = converter_optimized.convert()
    with open(OUTPUT_DIR / "fall_detection_1dcnn_dynamic_quant.tflite", "wb") as f:
        f.write(tflite_quant_model)


def print_classification_report(cm: np.ndarray) -> None:
    print("\nKlassenauswertung:")
    for class_id, class_name in enumerate(CLASS_NAMES):
        tp = cm[class_id, class_id]
        fp = cm[:, class_id].sum() - tp
        fn = cm[class_id, :].sum() - tp

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)

        print(
            f"{class_name:15s} "
            f"precision={precision:.3f} "
            f"recall={recall:.3f} "
            f"f1={f1:.3f}"
        )


def test_single_prediction(model: tf.keras.Model, x_test: np.ndarray, y_test: np.ndarray) -> None:
    idx = np.random.randint(0, len(y_test))
    sample = x_test[idx : idx + 1]
    probs = model.predict(sample, verbose=0)[0]

    print("\nBeispielvorhersage:")
    print(f"Echte Klasse: {CLASS_NAMES[int(y_test[idx])]}")
    for class_name, prob in zip(CLASS_NAMES, probs):
        print(f"  {class_name:15s}: {prob:.3f}")

In [8]:
# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main() -> None:
    print("TensorFlow:", tf.__version__)
    print(f"Output-Verzeichnis: {OUTPUT_DIR.resolve()}")

    x, y = generate_dataset(EXAMPLES_PER_CLASS)
    print("\nDataset:")
    print("x:", x.shape)
    print("y:", y.shape)

    x_train, y_train, x_val, y_val, x_test, y_test = split_dataset(x, y)

    mean, std = compute_normalization(x_train)
    x_train = normalize(x_train, mean, std)
    x_val = normalize(x_val, mean, std)
    x_test = normalize(x_test, mean, std)

    save_metadata(mean, std)

    model = build_model()
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
        ),
    ]

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=40,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    plot_history(history)

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    y_prob = model.predict(x_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    cm = confusion_matrix_np(y_test, y_pred, N_CLASSES)

    print("\nConfusion Matrix:")
    print(cm)
    print_classification_report(cm)
    plot_confusion_matrix(cm)

    model.save(OUTPUT_DIR / "fall_detection_1dcnn.keras")
    export_tflite(model)
    test_single_prediction(model, x_test, y_test)

    print("\nFertig. Erzeugte Dateien:")
    for file in sorted(OUTPUT_DIR.iterdir()):
        print(" -", file.name)

    print("\nWichtig für Android:")
    print("1. Nutze dieselbe Fenstergröße: 100 × 6")
    print("2. Nutze dieselbe Kanalreihenfolge: ax, ay, az, gx, gy, gz")
    print("3. Nutze mean/std aus metadata.json zur Normalisierung")
    print("4. Modell-Output ist eine Softmax-Wahrscheinlichkeit über die Klassen")


if __name__ == "__main__":
    # Weniger TensorFlow-Logspam.
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    main()


TensorFlow: 2.12.0-rc0
Output-Verzeichnis: C:\Users\aible\fall_detection_output
Erzeuge Klasse 0: alltag_ruhend
Erzeuge Klasse 1: gehen
Erzeuge Klasse 2: hinlegen
Erzeuge Klasse 3: handy_faellt
Erzeuge Klasse 4: sturz

Dataset:
x: (6000, 100, 6)
y: (6000,)
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_window (InputLayer)  [(None, 100, 6)]          0         
                                                                 
 conv1d (Conv1D)             (None, 100, 32)           992       
                                                                 
 batch_normalization (BatchN  (None, 100, 32)          128       
 ormalization)                                                   
                                                                 
 max_pooling1d (MaxPooling1D  (None, 50, 32)           0         
 )                                                               
  

INFO:tensorflow:Assets written to: C:\Users\aible\AppData\Local\Temp\tmprpvt4h6p\assets


INFO:tensorflow:Assets written to: C:\Users\aible\AppData\Local\Temp\tmprpvt4h6p\assets


INFO:tensorflow:Assets written to: C:\Users\aible\AppData\Local\Temp\tmpoi3qt4h6\assets


INFO:tensorflow:Assets written to: C:\Users\aible\AppData\Local\Temp\tmpoi3qt4h6\assets



Beispielvorhersage:
Echte Klasse: alltag_ruhend
  alltag_ruhend  : 1.000
  gehen          : 0.000
  hinlegen       : 0.000
  handy_faellt   : 0.000
  sturz          : 0.000

Fertig. Erzeugte Dateien:
 - confusion_matrix.png
 - fall_detection_1dcnn.keras
 - fall_detection_1dcnn_dynamic_quant.tflite
 - fall_detection_1dcnn_float32.tflite
 - metadata.json
 - training_accuracy.png
 - training_loss.png

Wichtig für Android:
1. Nutze dieselbe Fenstergröße: 100 × 6
2. Nutze dieselbe Kanalreihenfolge: ax, ay, az, gx, gy, gz
3. Nutze mean/std aus metadata.json zur Normalisierung
4. Modell-Output ist eine Softmax-Wahrscheinlichkeit über die Klassen
